In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')


In [ ]:

# Load the CSV file
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

print(f"Shape: {df_food.shape}")



In [ ]:
df_food.head()

In [ ]:
df_food.info()

In [ ]:
df_food.describe()

In [ ]:

# Target distribution
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()



In [ ]:
df_food1 = df_food.drop('Order_ID' , axis = 1)


In [ ]:
df_food1.head()

In [ ]:
print("\nMissing Values (df.isnull().sum()):")
print(df_food1.isnull().sum())
cols = {"Distance_km","Weather" ,"Traffic_Level" , 	"Time_of_Day", 	"Vehicle_Type", 	"Preparation_Time_min", 	"Courier_Experience_yrs", 	"Delivery_Time"}
df_clean = df_food1.dropna(subset=cols).copy()


print("\nMissing Values (df.isnull().sum()):")
print(df_clean.isnull().sum())

In [ ]:

print("Checking for duplicate rows...")
duplicate_rows = df_clean.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

In [ ]:


print("Separating target variable and features...")
X = df_clean.drop('Delivery_Time', axis=1)
y = df_clean['Delivery_Time']

print("Applying Label Encoding to target variable 'y'...")
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
print("Applying One-Hot Encoding to feature DataFrame 'X'...")
X_encoded = pd.DataFrame(onehot_encoder.fit_transform(X), columns=onehot_encoder.get_feature_names_out(X.columns))
print("Applying Label Encoding to target variable 'y'...")
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print("Verification of encoded data shapes:")
print(f"Shape of X_encoded: {X_encoded.shape}")

print("First 5 rows of X_encoded:")
display(X_encoded.head())
print("First 5 elements of y_encoded:")
print(y_encoded[:5])




In [ ]:
#  Scale features
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X_encoded)
#y_scaled = scaler.fit_transform(y_encoded)


In [ ]:
# Task 6: Write your code here:

In [ ]:


print("Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.3, random_state=42)

print("Data split successful.")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")



In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)


In [ ]:
def categorical_cross_entropy(y_true, y_pred):
    """Compute categorical cross-entropy loss."""
    epsilon = 1e-10 # Small value to prevent log(0)
    y_pred = np.clip(y_pred, epsilon, 1. - epsilon)
    loss = -np.sum(y_true * np.log(y_pred), axis=-1)
    return np.mean(loss)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, mean_absolute_error

In [ ]:

fold_losses = [] # List to store loss from each fold
for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
        # Split X_train and y_train into training and validation sets for the current fold
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        # Train the current model
        model.fit(X_train_fold, y_train_fold)

        # Generate predictions (probabilities) on the validation set
        y_pred_proba = model.predict_proba(X_val_fold)
        # Convert y_val_fold (true labels) into a one-hot encoded format
        y_val_one_hot = y_val_fold,

        # Calculate categorical cross-entropy loss for the current fold
        #loss = mean_absolute_error(y_val_one_hot, y_pred_proba)
        #fold_losses.append(loss)






In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

In [ ]:

    # Make predictions on the test set (hard labels)
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)


    # Print metrics
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")

    # Generate and visualize Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix for random forest')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: